# SecureSpeak — Publication-Readiness ExperimentsThree experiments that close the gaps a reviewer will raise about Paper 1.| # | Experiment | Gap it closes ||---|---|---|| 1 | Leakage audit + honest splits | "Random split inflates URL numbers; phishing evolves — where is the temporal/disjoint eval?" || 2 | MFS feature ablation | "SHAP shows the model *uses* MFS features; it does not show they *help*." || 3 | Decision-layer updatability | "The decision layer is described vaguely with no quantitative evaluation." |**Ground rules built into this notebook*** Every result prints with the split protocol that produced it. No number travels without its protocol.* Suspiciously perfect results (accuracy or AUC > 0.995) trigger an explicit warning, not a celebration.* Deltas come with bootstrap confidence intervals and a McNemar test. A delta without a CI is not a finding.* Assertions enforce no-overlap between train and test groups. If an assertion fires, the split was wrong.* Nothing here fabricates or imputes data. If an input is missing, the cell fails loudly instead of falling back.Run top to bottom. Cell 2 is the only cell you must edit.

## 0 · Configuration — **edit this cell only**

In [ ]:
# ============================================================# CONFIG# ============================================================# --- Path to your URL data -----------------------------------# Option A (preferred): a CSV that already contains your 26 engineered#   features, one column per feature, plus a URL column and a label column.# Option B: a CSV with just URLs + labels. Set REBUILD_FEATURES = True and#   the notebook will engineer the 26 features itself (see Section 1b).#   NOTE: rebuilt features will not reproduce your pipeline exactly.URL_CSV        = "/content/drive/MyDrive/SecureSpeak/stealthphisher_features.csv"REBUILD_FEATURES = FalseURL_COL        = "url"        # column holding the raw URL stringLABEL_COL      = "label"      # 1 = phishing, 0 = legitimateDATE_COL       = None         # e.g. "first_seen" if timestamps exist, else None# --- Which features are the MFS/Bangladesh knowledge features ---# These are the ones ablated in Experiment 2.MFS_FEATURES = [    "brand_impersonation",    "financial_kw",    "free_hosting_tld",    "high_risk_tld",]# --- MFS brand tokens (used for the brand-subset analysis) ------MFS_BRANDS = ["bkash", "nagad", "rocket", "upay", "dbbl", "sure cash", "surecash"]# --- Model + reproducibility ------------------------------------RANDOM_STATE  = 42TEST_SIZE     = 0.20N_BOOTSTRAP   = 1000RF_PARAMS     = dict(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)# --- Network / fusion data for Experiment 3 (optional) ----------# Leave as None to skip Experiment 3.FLOW_CSV      = None          # e.g. ".../cicmalanal_flows.csv"PORT_COL      = "dst_port"FLOW_LABEL    = "label"OUT_DIR       = "/content/securespeak_results"

In [ ]:
import os, sys, math, json, warnings, hashlibimport numpy as np, pandas as pdfrom collections import Counterfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.model_selection import train_test_split, GroupShuffleSplitfrom sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,                             precision_score, recall_score, confusion_matrix)from sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.neural_network import MLPClassifiernp.random.seed(RANDOM_STATE)os.makedirs(OUT_DIR, exist_ok=True)pd.set_option("display.width", 200)pd.set_option("display.max_columns", 60)try:    import tldextract    HAVE_TLDEXTRACT = Trueexcept ImportError:    HAVE_TLDEXTRACT = False    print("tldextract not installed. Run:  !pip install tldextract")    print("Falling back to a heuristic registered-domain parser.\n")RESULTS = {}   # every table produced in this notebook lands heredef flag_if_too_good(name, acc, auc):    """Sajid's rule: near-perfect numbers are a red flag, not a win."""    if acc is not None and acc > 0.995:        print(f"  !! {name}: accuracy {acc:.4f} exceeds 0.995 — audit for leakage before reporting.")    if auc is not None and auc > 0.999:        print(f"  !! {name}: AUC {auc:.4f} exceeds 0.999 — audit for leakage before reporting.")def evaluate(model, X_te, y_te, name=""):    pred = model.predict(X_te)    try:        proba = model.predict_proba(X_te)[:, 1]        auc = roc_auc_score(y_te, proba)    except Exception:        proba, auc = None, None    m = dict(        n_test    = int(len(y_te)),        accuracy  = accuracy_score(y_te, pred),        f1        = f1_score(y_te, pred, average="weighted"),        precision = precision_score(y_te, pred, zero_division=0),        recall    = recall_score(y_te, pred, zero_division=0),        auc       = auc,    )    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0,1]).ravel()    m["fnr"] = fn / (fn + tp) if (fn + tp) else float("nan")    m["fpr"] = fp / (fp + tn) if (fp + tn) else float("nan")    if name:        flag_if_too_good(name, m["accuracy"], m["auc"])    return m, preddef show(title, rows):    df = pd.DataFrame(rows)    print("\n" + "="*len(title)); print(title); print("="*len(title))    print(df.to_string(index=False))    return dfprint("Environment ready.")

## 1 · Load the data and audit it for leakageBefore any model is trained: how much of the test set is a near-copy of the training set?For URL corpora this is the single most common source of inflated accuracy — many phishingURLs share a registered domain, so a random split puts siblings of nearly every test URLinto training.

In [ ]:
df = pd.read_csv(URL_CSV)print(f"Loaded {len(df):,} rows, {df.shape[1]} columns from\n  {URL_CSV}\n")missing = [c for c in [URL_COL, LABEL_COL] if c not in df.columns]if missing:    raise KeyError(f"Missing required column(s) {missing}. Columns present: {list(df.columns)[:40]}")df[LABEL_COL] = df[LABEL_COL].astype(int)print("Label balance:")print(df[LABEL_COL].value_counts().rename({0:"legitimate",1:"phishing"}).to_string())print(f"\nPositive rate: {df[LABEL_COL].mean():.4f}")if DATE_COL and DATE_COL in df.columns:    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")    print(f"\nDate range: {df[DATE_COL].min()}  ->  {df[DATE_COL].max()}")    print(f"Rows with unparseable dates: {df[DATE_COL].isna().sum():,}")else:    print("\nNo DATE_COL configured — the temporal split in Section 1d will be skipped.")

In [ ]:
def registered_domain(url: str) -> str:    """Return the registrable domain (eTLD+1). Groups sibling URLs together."""    if not isinstance(url, str):        return ""    u = url.strip()    if HAVE_TLDEXTRACT:        ext = tldextract.extract(u)        return f"{ext.domain}.{ext.suffix}".lower().strip(".")    # heuristic fallback    u = u.split("://")[-1].split("/")[0].split("?")[0].split(":")[0]    parts = [p for p in u.lower().split(".") if p]    return ".".join(parts[-2:]) if len(parts) >= 2 else u.lower()df["_regdom"] = df[URL_COL].map(registered_domain)n_exact  = df[URL_COL].duplicated().sum()n_dom    = df["_regdom"].nunique()top_doms = df["_regdom"].value_counts().head(10)print(f"Exact duplicate URLs      : {n_exact:,}  ({n_exact/len(df):.2%})")print(f"Unique registered domains : {n_dom:,} across {len(df):,} URLs")print(f"Mean URLs per domain      : {len(df)/max(n_dom,1):.2f}")print("\nMost frequent domains:")print(top_doms.to_string())conc = top_doms.sum() / len(df)print(f"\nTop-10 domains account for {conc:.2%} of all rows.")if len(df)/max(n_dom,1) > 1.5:    print("\n>> Domains repeat across rows. A random split will leak sibling URLs")    print("   from train into test. Section 1c quantifies exactly how much this matters.")

### 1b · (Optional) rebuild the 26 features from raw URLsSkip if `REBUILD_FEATURES = False`. This exists so the notebook runs standalone and so thefeature definitions are explicit and publishable — one of the reproducibility gaps flaggedin review. Rebuilt features approximate, but will not exactly reproduce, your original pipeline.

In [ ]:
import refrom urllib.parse import urlparseHIGH_RISK_TLDS   = {"zip","review","country","kim","cricket","science","work","party","gq","link"}FREE_HOST_TLDS   = {"tk","ml","ga","cf","gq"}FINANCIAL_KW     = ["bank","pay","wallet","cash","account","login","secure","verify","transfer",                    "balance","otp","pin","fund","transaction","refund","bonus"]URGENCY_KW       = ["urgent","immediate","expire","suspend","block","alert","warning","now",                    "confirm","update","limited","win","prize","offer"]def shannon_entropy(s: str) -> float:    if not s: return 0.0    counts = Counter(s)    n = len(s)    return -sum((c/n) * math.log2(c/n) for c in counts.values())def build_features(url: str) -> dict:    u  = url if isinstance(url, str) else ""    lo = u.lower()    try:        parsed = urlparse(u if "://" in u else "http://" + u)    except Exception:        parsed = urlparse("http://invalid")    host = (parsed.netloc or "").lower()    path = parsed.path or ""    host_nport = host.split(":")[0]    labels = [p for p in host_nport.split(".") if p]    tld = labels[-1] if labels else ""    regdom = registered_domain(u)    core   = regdom.split(".")[0] if regdom else ""    # brand token appearing anywhere EXCEPT as the registrable core = impersonation    brand_hits = 0    for b in MFS_BRANDS:        bn = b.replace(" ", "")        if bn in lo.replace(" ", "") and bn != core:            brand_hits += 1    f = {        # lexical structure        "url_length"        : len(u),        "hostname_length"   : len(host_nport),        "path_length"       : len(path),        "num_dots"          : u.count("."),        "num_hyphens"       : u.count("-"),        "num_digits"        : sum(ch.isdigit() for ch in u),        "num_slashes"       : u.count("/"),        "num_special_chars" : sum(ch in "@?=&%_~+" for ch in u),        "num_subdomains"    : max(len(labels) - 2, 0),        "has_at_symbol"     : int("@" in u),        "has_ip_address"    : int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host_nport))),        # entropy        "url_entropy"       : shannon_entropy(u),        "domain_entropy"    : shannon_entropy(host_nport),        # domain risk        "high_risk_tld"     : int(tld in HIGH_RISK_TLDS),        "free_hosting_tld"  : int(tld in FREE_HOST_TLDS),        "is_https"          : int(parsed.scheme == "https"),        "has_port"          : int(":" in host),        "domain_length"     : len(regdom),        "num_query_params"  : len([q for q in (parsed.query or "").split("&") if q]),        # Bangladesh / MFS-specific knowledge features        "brand_impersonation": brand_hits,        "financial_kw"      : sum(k in lo for k in FINANCIAL_KW),        "urgency_kw"        : sum(k in lo for k in URGENCY_KW),        # structural        "has_hex_encoding"  : int("%" in u),        "digit_ratio"       : (sum(ch.isdigit() for ch in u) / len(u)) if u else 0.0,        "longest_token_len" : max([len(t) for t in re.split(r"[./\-_?=&]", u) if t] or [0]),        "path_depth"        : len([p for p in path.split("/") if p]),    }    return fif REBUILD_FEATURES:    feats = pd.DataFrame([build_features(u) for u in df[URL_COL]])    print(f"Engineered {feats.shape[1]} features for {len(feats):,} URLs.")    df = pd.concat([df.drop(columns=[c for c in feats.columns if c in df.columns]), feats], axis=1)else:    print("REBUILD_FEATURES = False — using the feature columns already in your CSV.")FEATURE_COLS = [c for c in df.columns                if c not in {URL_COL, LABEL_COL, "_regdom"}                and (DATE_COL is None or c != DATE_COL)                and pd.api.types.is_numeric_dtype(df[c])]print(f"\nUsing {len(FEATURE_COLS)} feature columns.")absent = [f for f in MFS_FEATURES if f not in FEATURE_COLS]if absent:    raise KeyError(f"MFS_FEATURES not found in the data: {absent}\n"                   f"Edit MFS_FEATURES in the config to match your column names.\n"                   f"Available: {FEATURE_COLS}")print(f"MFS knowledge features confirmed present: {MFS_FEATURES}")

### 1c · Three splits, same modelThe only thing that changes between these three runs is **how the data is divided**.Any gap between them is leakage, not learning.* **Random** — what the paper currently reports.* **Domain-disjoint** — no registered domain appears in both train and test. This is the URL  analogue of the family-level holdout already used for the network detector. Applying that  rigor to malware but not to URLs is the inconsistency a reviewer will notice.* **Temporal** — train on the past, test on the future. Only runs if `DATE_COL` is set.

In [ ]:
X   = df[FEATURE_COLS].valuesy   = df[LABEL_COL].valuesgrp = df["_regdom"].valuesdef make_pipeline():    # scaler INSIDE the pipeline: normalization statistics never see the test fold    return Pipeline([("scale", StandardScaler()),                     ("clf",   RandomForestClassifier(**RF_PARAMS))])split_rows = []split_store = {}# ---- Random split -------------------------------------------------Xtr, Xte, ytr, yte, gtr, gte = train_test_split(    X, y, grp, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)pipe = make_pipeline().fit(Xtr, ytr)m, _ = evaluate(pipe, Xte, yte, name="Random split")overlap = len(set(gtr) & set(gte))m.update(split="Random (as reported)", domain_overlap=overlap)split_rows.append(m); split_store["random"] = (Xtr, Xte, ytr, yte)print(f"Random split: {overlap:,} registered domains appear in BOTH train and test.")# ---- Domain-disjoint split ---------------------------------------gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)tr_i, te_i = next(gss.split(X, y, groups=grp))Xtr2, Xte2, ytr2, yte2 = X[tr_i], X[te_i], y[tr_i], y[te_i]assert len(set(grp[tr_i]) & set(grp[te_i])) == 0, "Domain-disjoint split failed — overlap detected"pipe2 = make_pipeline().fit(Xtr2, ytr2)m2, _ = evaluate(pipe2, Xte2, yte2, name="Domain-disjoint split")m2.update(split="Domain-disjoint", domain_overlap=0)split_rows.append(m2); split_store["disjoint"] = (Xtr2, Xte2, ytr2, yte2)print("Domain-disjoint split: 0 domain overlap (asserted).")# ---- Temporal split ----------------------------------------------if DATE_COL and DATE_COL in df.columns and df[DATE_COL].notna().sum() > 100:    d = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL)    cut = int(len(d) * (1 - TEST_SIZE))    tr, te = d.iloc[:cut], d.iloc[cut:]    Xtr3, ytr3 = tr[FEATURE_COLS].values, tr[LABEL_COL].values    Xte3, yte3 = te[FEATURE_COLS].values, te[LABEL_COL].values    pipe3 = make_pipeline().fit(Xtr3, ytr3)    m3, _ = evaluate(pipe3, Xte3, yte3, name="Temporal split")    m3.update(split=f"Temporal (train<{te[DATE_COL].min().date()})",              domain_overlap=len(set(tr['_regdom']) & set(te['_regdom'])))    split_rows.append(m3); split_store["temporal"] = (Xtr3, Xte3, ytr3, yte3)    print(f"Temporal split: train n={len(tr):,}, test n={len(te):,}")else:    print("Temporal split skipped (no usable DATE_COL).")cols = ["split","n_test","accuracy","f1","auc","recall","fnr","fpr","domain_overlap"]tbl1 = show("TABLE — URL detector under three split protocols",            [{k: r.get(k) for k in cols} for r in split_rows])RESULTS["split_protocols"] = tbl1tbl1.to_csv(f"{OUT_DIR}/exp1_splits.csv", index=False)

In [ ]:
# Interpretation guide — read this before writing up Experiment 1r = RESULTS["split_protocols"].set_index("split")rand_acc = r.iloc[0]["accuracy"]dis_acc  = r.loc["Domain-disjoint", "accuracy"]drop     = rand_acc - dis_accprint(f"Random split accuracy          : {rand_acc:.4f}")print(f"Domain-disjoint accuracy       : {dis_acc:.4f}")print(f"Drop attributable to leakage   : {drop:.4f}  ({drop/rand_acc:.2%} relative)\n")if drop > 0.05:    print("LARGE DROP. The headline number was substantially driven by sibling-URL leakage.")    print("Report the domain-disjoint number as the primary result. This is the honest number,")    print("and reporting it is a strength, not a weakness — it is exactly what TESSERACT argues for.")elif drop > 0.01:    print("MODERATE DROP. Report both protocols side by side in Table 1.")    print("The gap itself is a finding worth one paragraph.")else:    print("SMALL DROP. The detector genuinely generalizes across unseen domains.")    print("Report both anyway — a reviewer who sees you tested for leakage and found none")    print("trusts every other number in the paper more.")

## 2 · MFS feature ablationThe paper's central intellectual claim is that encoding MFS-specific knowledge as featuresmakes the detector better for the Bangladeshi threat context. SHAP attribution shows the model*uses* those features. It does not show they *help*. This is what shows whether they help.Two test sets are evaluated:* the **full** held-out set, where the effect will be diluted, and* the **MFS-brand subset** — test URLs containing an MFS brand token — where the effect should  concentrate if the claim is true.Deltas are reported with a bootstrap 95% CI and a McNemar test, because a difference of a fewhundredths of a percent at this accuracy level is meaningless without one.

In [ ]:
def mcnemar_test(y_true, pred_a, pred_b):    """Exact McNemar on the discordant pairs. Returns (b, c, p_value)."""    a_ok = (pred_a == y_true); b_ok = (pred_b == y_true)    n01 = int(np.sum(a_ok & ~b_ok))   # A right, B wrong    n10 = int(np.sum(~a_ok & b_ok))   # A wrong, B right    n = n01 + n10    if n == 0:        return n01, n10, 1.0    from scipy.stats import binomtest    p = binomtest(min(n01, n10), n=n, p=0.5).pvalue    return n01, n10, pdef bootstrap_delta(y_true, pred_full, pred_abl, metric=accuracy_score, n_boot=N_BOOTSTRAP):    """95% CI on (full - ablated) via paired bootstrap over test indices."""    rng = np.random.default_rng(RANDOM_STATE)    n = len(y_true); deltas = np.empty(n_boot)    for i in range(n_boot):        idx = rng.integers(0, n, n)        deltas[i] = metric(y_true[idx], pred_full[idx]) - metric(y_true[idx], pred_abl[idx])    return float(np.mean(deltas)), float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5))# Choose the split protocol the ablation runs on.# Domain-disjoint is the defensible choice — run the ablation on the honest split.ABL_SPLIT = "disjoint" if "disjoint" in split_store else "random"Xtr_a, Xte_a, ytr_a, yte_a = split_store[ABL_SPLIT]print(f"Ablation runs on the '{ABL_SPLIT}' split (n_train={len(ytr_a):,}, n_test={len(yte_a):,}).\n")mfs_idx    = [FEATURE_COLS.index(f) for f in MFS_FEATURES]keep_idx   = [i for i in range(len(FEATURE_COLS)) if i not in mfs_idx]print(f"Full model  : {len(FEATURE_COLS)} features")print(f"Ablated     : {len(keep_idx)} features (removed {MFS_FEATURES})")m_full = make_pipeline().fit(Xtr_a, ytr_a)m_abl  = make_pipeline().fit(Xtr_a[:, keep_idx], ytr_a)res_full, pred_full = evaluate(m_full, Xte_a,             yte_a, name="Full model")res_abl,  pred_abl  = evaluate(m_abl,  Xte_a[:, keep_idx], yte_a, name="Ablated model")

In [ ]:
# --- identify the MFS-brand subset of the test set -------------------if ABL_SPLIT == "disjoint":    te_mask_idx = te_ielse:    # reconstruct indices for the random split    _, te_pos = train_test_split(np.arange(len(df)), test_size=TEST_SIZE,                                 random_state=RANDOM_STATE, stratify=y)    te_mask_idx = te_postest_urls = df.iloc[te_mask_idx][URL_COL].astype(str).str.lower().valuesbrand_mask = np.array([any(b.replace(" ","") in u.replace(" ","") for b in MFS_BRANDS)                       for u in test_urls])print(f"MFS-brand URLs in test set: {brand_mask.sum():,} of {len(brand_mask):,} "      f"({brand_mask.mean():.2%})")rows = []def add_row(label, y_t, p_f, p_a):    if len(y_t) < 30:        print(f"  (skipping '{label}' — only {len(y_t)} samples, too few to report)")        return    acc_f, acc_a = accuracy_score(y_t, p_f), accuracy_score(y_t, p_a)    rec_f, rec_a = recall_score(y_t, p_f, zero_division=0), recall_score(y_t, p_a, zero_division=0)    d, lo, hi    = bootstrap_delta(y_t, p_f, p_a)    n01, n10, p  = mcnemar_test(y_t, p_f, p_a)    rows.append(dict(subset=label, n=len(y_t),                     acc_full=round(acc_f,4), acc_ablated=round(acc_a,4),                     delta_acc=round(acc_f-acc_a,4),                     ci95=f"[{lo:+.4f}, {hi:+.4f}]",                     recall_full=round(rec_f,4), recall_ablated=round(rec_a,4),                     mcnemar_p=f"{p:.2e}" if p < 0.01 else f"{p:.3f}"))add_row("Full test set", yte_a, pred_full, pred_abl)if brand_mask.sum() >= 30:    add_row("MFS-brand subset", yte_a[brand_mask], pred_full[brand_mask], pred_abl[brand_mask])tbl2 = show("TABLE — ablation of MFS knowledge features", rows)RESULTS["ablation"] = tbl2tbl2.to_csv(f"{OUT_DIR}/exp2_ablation.csv", index=False)

In [ ]:
# Interpretation guide — read this before writing up Experiment 2a = RESULTS["ablation"]full_row  = a[a.subset == "Full test set"].iloc[0]brand_row = a[a.subset == "MFS-brand subset"].iloc[0] if (a.subset == "MFS-brand subset").any() else Noneprint(f"Full test set delta      : {full_row.delta_acc:+.4f}   CI {full_row.ci95}   p={full_row.mcnemar_p}")if brand_row is not None:    print(f"MFS-brand subset delta   : {brand_row.delta_acc:+.4f}   CI {brand_row.ci95}   p={brand_row.mcnemar_p}")print()def ci_excludes_zero(ci):     lo, hi = [float(x) for x in ci.strip("[]").split(",")]    return lo > 0 or hi < 0if brand_row is not None and brand_row.delta_acc > full_row.delta_acc and ci_excludes_zero(brand_row.ci95):    print("BEST CASE FOR THE PAPER. The MFS features contribute little on average but")    print("substantially on brand-impersonation attacks — which is precisely the argument.")    print("Write it that way: the knowledge features are not a general accuracy trick,")    print("they are targeted coverage for the local threat, and here is the number.")elif ci_excludes_zero(full_row.ci95) and full_row.delta_acc > 0:    print("The MFS features help significantly overall. Report the delta with its CI in Table 1")    print("and keep the claim proportional to the effect size.")else:    print("NO SIGNIFICANT BENEFIT. This is important to know before a reviewer finds it.")    print("Do not claim the MFS features improve accuracy. The honest and still-publishable")    print("framing is that they provide INTERPRETABILITY — they let the system say *why* a")    print("link is dangerous in a warning a non-expert can act on — at no cost to accuracy.")    print("That is a real contribution. An unsupported accuracy claim is not.")

## 3 · Decision-layer updatabilityThe reviewer's complaint that the decision layer is "described vaguely, no quantitative eval"is fair. This turns its central property into a measured result.**Setup.** A malicious port is designated "new" and every attack flow using it is removed fromtraining. Then:* the **learned fusion model** sees the new-port attacks having never trained on them, and* the **rule-augmented decision layer** gets one line added to its context rules — no retraining,  no redeployment — and is re-evaluated.This is a capability claim, not an accuracy claim, and should be written up as one: the rulelayer adapts in the time it takes to push a config change; the learned model cannot adapt at allwithout a full retrain cycle.

In [ ]:
if FLOW_CSV is None:    print("FLOW_CSV is None — Experiment 3 skipped.")    print("Set FLOW_CSV in the config to run it.")else:    fl = pd.read_csv(FLOW_CSV)    print(f"Loaded {len(fl):,} flows.")    for c in [PORT_COL, FLOW_LABEL]:        if c not in fl.columns:            raise KeyError(f"'{c}' not in flow data. Columns: {list(fl.columns)[:40]}")    fl[FLOW_LABEL] = fl[FLOW_LABEL].astype(int)    atk_ports = fl.loc[fl[FLOW_LABEL] == 1, PORT_COL].value_counts()    print("\nMost common attack destination ports:")    print(atk_ports.head(10).to_string())    # pick a port with enough attack volume to measure, but not the dominant one    candidates = atk_ports[(atk_ports >= 50)]    if len(candidates) < 2:        raise ValueError("Not enough distinct attack ports with >=50 flows to run this cleanly.")    NEW_PORT = int(candidates.index[min(1, len(candidates)-1)])    print(f"\nDesignating port {NEW_PORT} as the 'new' attack port "          f"({int(candidates.loc[NEW_PORT]):,} attack flows withheld).")    ffeats = [c for c in fl.columns              if c not in {FLOW_LABEL} and pd.api.types.is_numeric_dtype(fl[c])]    is_new = (fl[PORT_COL] == NEW_PORT) & (fl[FLOW_LABEL] == 1)    train_df = fl[~is_new]    newatk   = fl[is_new]    assert not ((train_df[PORT_COL] == NEW_PORT) & (train_df[FLOW_LABEL] == 1)).any(), \        "New-port attacks leaked into training"    print(f"Train: {len(train_df):,} flows  |  Withheld new-port attacks: {len(newatk):,}")    Xf, yf = train_df[ffeats].values, train_df[FLOW_LABEL].values    Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(        Xf, yf, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=yf)    mlp = Pipeline([("scale", StandardScaler()),                    ("clf", MLPClassifier(hidden_layer_sizes=(64,32), max_iter=400,                                          random_state=RANDOM_STATE))]).fit(Xf_tr, yf_tr)    Xnew = newatk[ffeats].values    ynew = np.ones(len(newatk), dtype=int)    mlp_recall = recall_score(ynew, mlp.predict(Xnew), zero_division=0)    # Rule-augmented layer: MLP score OR context rule on the newly-known malicious port    MALICIOUS_PORTS = set()                       # before the update    def rule_layer(frame, mlp_model, ports):        base = mlp_model.predict(frame[ffeats].values)        rule = frame[PORT_COL].isin(ports).values.astype(int)        return np.maximum(base, rule)    before = recall_score(ynew, rule_layer(newatk, mlp, MALICIOUS_PORTS), zero_division=0)    MALICIOUS_PORTS.add(NEW_PORT)                 # <-- the entire "update"    after  = recall_score(ynew, rule_layer(newatk, mlp, MALICIOUS_PORTS), zero_division=0)    # cost of the update on benign traffic    benign = fl[fl[FLOW_LABEL] == 0]    fpr_before = (rule_layer(benign, mlp, set()) == 1).mean()    fpr_after  = (rule_layer(benign, mlp, MALICIOUS_PORTS) == 1).mean()    rows3 = [        dict(system="Learned fusion (MLP), no retraining",             new_port_recall=round(mlp_recall,4), benign_fpr=round(fpr_before,4),             adapts_without_retraining="No"),        dict(system="Decision layer, before rule update",             new_port_recall=round(before,4), benign_fpr=round(fpr_before,4),             adapts_without_retraining="-"),        dict(system="Decision layer, after one-line rule update",             new_port_recall=round(after,4), benign_fpr=round(fpr_after,4),             adapts_without_retraining="Yes"),    ]    tbl3 = show(f"TABLE — adaptation to a new attack port ({NEW_PORT}), no retraining", rows3)    RESULTS["updatability"] = tbl3    tbl3.to_csv(f"{OUT_DIR}/exp3_updatability.csv", index=False)    print(f"\nCost of the update: benign false-positive rate moved "          f"{fpr_before:.4f} -> {fpr_after:.4f} ({fpr_after-fpr_before:+.4f}).")    print("Report this cost alongside the gain. An adaptation mechanism that silently")    print("raises false positives on benign traffic is not free, and saying so is the")    print("difference between an honest capability claim and marketing.")

## 4 · Export — paste-ready tablesEverything above, written out as CSV and as LaTeX `booktabs` tables.

In [ ]:
def to_latex(df, caption, label):    body = df.to_latex(index=False, escape=True, column_format="l" + "r"*(df.shape[1]-1))    return ("\\begin{table}[t]\n\\centering\n\\small\n"            + body +            f"\\caption{{{caption}}}\n\\label{{tab:{label}}}\n\\end{{table}}\n")meta = {    "n_rows": int(len(df)),    "n_features": len(FEATURE_COLS),    "mfs_features": MFS_FEATURES,    "random_state": RANDOM_STATE,    "test_size": TEST_SIZE,    "n_bootstrap": N_BOOTSTRAP,    "rf_params": {k: str(v) for k, v in RF_PARAMS.items()},    "unique_registered_domains": int(df["_regdom"].nunique()),    "ablation_split": ABL_SPLIT,}with open(f"{OUT_DIR}/run_metadata.json", "w") as fh:    json.dump(meta, fh, indent=2)captions = {    "split_protocols": ("URL detector under three split protocols. The gap between random and "                        "domain-disjoint quantifies sibling-URL leakage.", "splits"),    "ablation":        ("Ablation of the MFS knowledge features, with paired-bootstrap 95\\% CIs "                        "and exact McNemar tests.", "ablation"),    "updatability":    ("Adaptation to a previously unseen attack port without retraining.", "update"),}tex = []for key, tdf in RESULTS.items():    cap, lab = captions.get(key, (key, key))    tex.append(to_latex(tdf, cap, lab))latex_all = "\n".join(tex)with open(f"{OUT_DIR}/tables.tex", "w") as fh:    fh.write(latex_all)print(f"Written to {OUT_DIR}/:")for f in sorted(os.listdir(OUT_DIR)):    print("  ", f)print("\n" + "="*60)print(latex_all)

## 5 · What to do with the results**Experiment 1.** Whatever the domain-disjoint number is, that becomes the primary reportedresult and the random-split number moves beside it as a leakage diagnostic. If the drop islarge, that is not a failure — a paper that reports 94% under a disjoint split is morecredible than one reporting 99.76% under a split the reviewer suspects. Applying the sameholdout rigor to URLs that you already apply to malware families also removes theinconsistency with the TESSERACT protocol you cite.**Experiment 2.** Three outcomes, three honest write-ups:* Large effect on the brand subset, small overall → the paper's thesis, now measured. Best case.* Significant overall → state the delta with its CI; keep the claim proportional.* No significant effect → drop the accuracy claim entirely and reframe the MFS features as an  *interpretability* contribution. They let the system name the reason a link is dangerous,  which is what makes an actionable bilingual warning possible. That is a genuine contribution  and it survives review. An unsupported accuracy claim does not.**Experiment 3.** Report the gain *and* the false-positive cost together. The claim is"adapts without retraining," never "more accurate than a learned model."**Still outstanding, in priority order**1. The user study with elderly and low-literacy MFS users. This is Paper 2, and it is what   moves the work from a good systems paper to a CHI/SOUPS-tier contribution.2. A public repository with these scripts, hyperparameters, and the feature definitions.3. The brand-rescue experiment — the knowledge-layer detector evaluated on a curated set of   real MFS-brand phishing URLs.**One correction to make in the paper regardless of any result here.** The comparison againstTanbhir et al. (ICECE 2024), who report 98.47% on BangalaBarta with a BERT + character-levelCNN, must be added to Section 4.2 and Related Work. They are the direct baseline on the samecorpus, and the honest framing is favourable: SecureSpeak reaches comparable accuracy with aMiniLM encoder plus logistic regression, at a fraction of the inference cost, which is whatmakes on-device deployment on mid-range Android hardware feasible.